# Disclosure risk against the public register

The attacker has the released file, the public register and our report.

1. **Identity:** can a record be linked to a person in the register? For each record, `F` = number of register people with the same quasi-identifiers (an empty, suppressed cell matches anyone). Risk = `1/F`.
2. **Party:** starting from a respondent's register entry, can the attacker tell how they voted?

Computed for the three versions: the **raw data**, the data **after recoding only**, and the **released file**.

In [ ]:
import pandas as pd, numpy as np
from datetime import date

DATE_THEN = date(2025, 7, 1)
QI     = ["sex", "evote", "zip", "citizenship", "age_group", "marital_status"]   # in the register (education is not)
RAW_QI = ["sex", "evote", "zip", "citizenship", "dob", "marital_status"]

## Load data

In [ ]:
private  = pd.read_excel(r"..\data\private_dataF.xlsx")
register = pd.read_excel(r"..\data\public_data_registerF.xlsx")
recoded  = pd.read_csv(r"..\data\private_data_preprocessed.csv", dtype=str, index_col=0)
released = pd.read_csv(r"..\data\anonymised_dataF.csv", dtype=str)

# raw data as text so it compares with the register
raw = private.astype(str)
raw["dob"] = pd.to_datetime(private["dob"], format="%d/%m/%Y")

# true names, in the same row order (preprocessing only dropped the 4 'Not stated' rows)
names_raw = private["name"].reset_index(drop=True)
names     = private.loc[private["education"] != "Not stated", "name"].reset_index(drop=True)
assert (private.loc[private["education"] != "Not stated", "party"].values == released["party"].values).all()

## Register, recoded like the data
Abstainers (`last_voted = 2`) cannot be respondents, so they are removed.

In [ ]:
reg = register[register["last_voted"] != 2].astype(str).rename(columns={"last_voted": "evote"})
reg["dob"] = pd.to_datetime(register.loc[reg.index, "dob"], format="%d/%m/%Y")
age = reg["dob"].apply(lambda d: DATE_THEN.year - d.year - ((DATE_THEN.month, DATE_THEN.day) < (d.month, d.day)))

reg_raw = reg
reg_rec = reg.assign(
    age_group   = pd.cut(age, [18, 30, 45, 65, np.inf], right=False,
                         labels=["18-29", "30-44", "45-64", "65 or older"]).astype(str),
    citizenship = np.where(reg["citizenship"] == "Denmark", "Danish", "non-danish"))
reg_rel = reg_rec.assign(zip=reg_rec["zip"].str[0] + "***",   # released file: ZIP as 2***, marital merged
                         marital_status=reg_rec["marital_status"].where(reg_rec["marital_status"] == "Never married", "Ever married"))

# (data, register view, quasi-identifiers, true names)
VERSIONS = {
    "raw data":            (raw,      reg_raw, RAW_QI, names_raw),
    "after recoding only": (recoded,  reg_rec, QI,     names),
    "released file":       (released, reg_rel, QI,     names),
}
print("register people who voted:", len(reg))

## Risk metrics
- **unique in register (F = 1):** the record points to exactly one person, i.e. re-identified
- **smallest F:** k-anonymity measured against the register
- **mean risk:** average of `1/F`; times the number of records = expected re-identifications
- **party certain:** all records compatible with the respondent's register entry have the same party
- **party guess accuracy:** attacker picks the most common party among the compatible records

In [ ]:
def matches(row, table, qi):
    """Rows of `table` that agree with `row` on every quasi-identifier; empty cells match anything."""
    m = pd.Series(True, index=table.index)
    for q in qi:
        if pd.notna(row[q]):
            m &= table[q].isna() | (table[q] == row[q])
    return table[m]

def risk(data, reg_view, qi, true_names):
    F = np.array([len(matches(rec, reg_view, qi)) for _, rec in data.iterrows()])

    # party: look up each respondent in the register, then find the records they could be
    who = reg_view.set_index("name").loc[true_names]
    certain, correct = 0, 0
    for (_, person), party in zip(who.iterrows(), data["party"]):
        shares = matches(person, data, qi)["party"].value_counts(normalize=True)
        certain += shares.iloc[0] == 1
        correct += shares.index[0] == party

    return pd.Series({
        "records":                 len(data),
        "unique in register (F=1)": int((F == 1).sum()),
        "smallest F":              int(F.min()),
        "mean risk (1/F)":         round((1 / F).mean(), 3),
        "expected re-identifications": round((1 / F).sum(), 1),
        "party certain":           int(certain),
        "party guess accuracy":    round(correct / len(data), 3),
    })

print("baseline: always guessing the most common party =",
      round(released["party"].value_counts(normalize=True).iloc[0], 3))
pd.DataFrame({k: risk(*v) for k, v in VERSIONS.items()})